# `risk_classifier_bundle.pkl` — Model Reference

What it is, how to load it, and exactly what inputs/outputs each function
needs. Written for whoever wires this into the backend, without needing to
read the notebook.

---

## What this file is

A single pickled Python dictionary containing a trained classifier plus
everything needed to actually use it — the encoders that turn human-readable
inputs into the numbers the model expects, and a set of per-hotspot
statistics for anomaly detection. It is **not** a model file in the
TensorFlow/ONNX sense — it's a plain `pickle.dump()` of a Python dict, one
of whose values happens to be a scikit-learn model object.

Produced by: `4_tier2_tier3_combined.ipynb` (or `5.ipynb`), final cell of
the "Persist the trained model" section. Trained on **300 real clusters**
from your team's actual DBSCAN clustering output.

**Source of truth:** the field names and shapes below were re-read directly
from the notebook's source code (not from memory), to avoid documentation
drift. If the notebook changes (e.g. feature list, bucket names), this file
will go stale — re-generate it from the notebook's current code if so.

---

## Loading it

```python
import pickle

with open("risk_classifier_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)
```

Load this **once**, at backend startup — not per-request. Loading is the
slow part (deserializing the trained forest); using it afterward is fast.

### What's inside `bundle`

| Key | Type | What it is |
|---|---|---|
| `bundle["classifier"]` | `sklearn.ensemble.RandomForestClassifier` | The trained Tier 2 HIGH/LOW model |
| `bundle["day_map"]` | `dict` | `{"Monday": 0, "Tuesday": 1, ..., "Sunday": 6}` |
| `bundle["bucket_map"]` | `dict` | `{"early_morning": 0, "late_morning": 1, "afternoon": 2, "evening": 3, "night": 4}` |
| `bundle["features"]` | `list[str]` | `["day_encoded", "bucket_encoded", "congestion_impact_score", "total_incident_count"]` — exact column order the model expects |
| `bundle["anomaly_baselines"]` | `dict` | `{cluster_id: {"baseline_mean": float, "baseline_std": float, "n_days_observed": int}, ...}` — one entry per cluster |
| `bundle["anomaly_threshold"]` | `float` | `2.5` — the z-score cutoff used to call a day "anomalous" |
| `bundle["trained_on_n_clusters"]` | `int` | `300` — sanity/versioning check |

**Never use `day_map`/`bucket_map` from anywhere else** (e.g. don't
hand-write your own day-name-to-int mapping in the backend). If the
encoding doesn't match exactly what the model was trained on, predictions
will be silently wrong, not an error.

---

## Function 1: `predict_priority_risk` — Tier 2, HIGH/LOW prediction

Answers: *"is this hotspot, on this day of the week, at this hour,
normally a high-risk slot?"*

### Inputs (exact format)

| Argument | Type | Valid values | Example |
|---|---|---|---|
| `cluster_id` | `int` | One of the 300 trained cluster IDs (see note below on how to get a valid list) | `2` |
| `day_of_week` | `str` | Exactly one of: `"Monday"`, `"Tuesday"`, `"Wednesday"`, `"Thursday"`, `"Friday"`, `"Saturday"`, `"Sunday"` — capitalized, full name, no abbreviations | `"Monday"` |
| `hour` | `int` | `0`–`23` (24-hour clock) | `10` |

`hour` is automatically converted internally into one of 5 time buckets —
you do not need to pre-bucket it yourself:

| Hour range | Bucket name |
|---|---|
| 3–6 | `early_morning` |
| 7–11 | `late_morning` |
| 12–15 | `afternoon` |
| 16–19 | `evening` |
| 0, 1, 2, 20, 21, 22, 23 | `night` |

### How to call it (since the function as written in the notebook closes
over notebook-local variables, here's the backend-safe standalone version)

```python
import pandas as pd

def predict_priority_risk(cluster_id, day_of_week, hour, bundle, cluster_summary):
    """
    cluster_summary: the cluster_summary_checkpoint.csv DataFrame, loaded
    once at startup alongside the pickle. Needed because congestion_impact_score
    and total_incident_count are per-cluster lookups, not part of the pickle.
    """
    clf = bundle["classifier"]
    day_map = bundle["day_map"]
    bucket_map = bundle["bucket_map"]
    features = bundle["features"]

    def coarse_bucket(h):
        if 3 <= h < 7: return "early_morning"
        if 7 <= h < 12: return "late_morning"
        if 12 <= h < 16: return "afternoon"
        if 16 <= h < 20: return "evening"
        return "night"

    row = cluster_summary[cluster_summary["cluster_id"] == cluster_id]
    if row.empty:
        return {"cluster_id": cluster_id, "error": "cluster not found"}
    if day_of_week not in day_map:
        return {"cluster_id": cluster_id, "error": f"invalid day_of_week, expected one of {list(day_map.keys())}"}

    bucket = coarse_bucket(hour)
    feature_row = pd.DataFrame([{
        "day_encoded": day_map[day_of_week],
        "bucket_encoded": bucket_map[bucket],
        "congestion_impact_score": row["congestion_impact_score"].values[0],
        "total_incident_count": row["total_incident_count"].values[0],
    }])[features]

    pred_label = clf.predict(feature_row)[0]
    pred_proba = clf.predict_proba(feature_row)[0]

    return {
        "cluster_id": int(cluster_id),
        "day_of_week": day_of_week,
        "hour": int(hour),
        "time_bucket": bucket,
        "predicted_risk": "HIGH" if pred_label == 1 else "LOW",
        "confidence": round(float(max(pred_proba)), 3),
    }
```

### Output (exact shape)

```json
{
  "cluster_id": 2,
  "day_of_week": "Monday",
  "hour": 10,
  "time_bucket": "late_morning",
  "predicted_risk": "HIGH",
  "confidence": 0.861
}
```

On bad input, returns an error dict instead of raising — e.g.
`{"cluster_id": 99999, "error": "cluster not found"}`. Always check for an
`"error"` key before reading `"predicted_risk"`.

### What it actually tells you

`confidence` is the model's probability for whichever class it picked
(not specifically the probability of HIGH) — e.g. `confidence: 0.861` with
`predicted_risk: "LOW"` means 86.1% confident it's LOW, not HIGH.

**Important honesty note:** since the only inputs are `day_of_week` and
`hour` (mapped to a bucket) plus two numbers that are constant for a given
cluster, there are only `300 × 7 × 5 = 10,500` possible distinct outputs
this function can ever produce, total. It will return the exact same
answer for the same `(cluster_id, day_of_week, hour-bucket)` every single
time — it is not reacting to anything happening live. All 10,500 of these
were already precomputed into `hotspots_details.json`'s
`priority_prediction` field. Calling this function live is genuine model
inference (useful to demo that a real model exists, not just a lookup
table), but it will not produce a new answer the static JSON doesn't
already have.

---

## Function 2: `check_anomaly` — Tier 3, anomaly detection

Answers: *"was this specific date's violation count unusual for this
specific hotspot?"* — fundamentally different from Tier 2 because it
needs an **already-observed count**, not just a day/hour. This is the
function that can produce a genuinely new answer live (see note at the
end).

### Inputs (exact format)

| Argument | Type | Valid values | Example |
|---|---|---|---|
| `cluster_id` | `int` | One of the 300 trained cluster IDs | `0` |
| `date` | `datetime.date` (or an ISO string `"YYYY-MM-DD"`, see note) | Any calendar date | `2023-12-22` |

### How to call it (standalone version)

```python
def check_anomaly(cluster_id, date, actual_count, bundle):
    """
    actual_count: the OBSERVED violation count for this cluster on this
    date. The original notebook version looks this up from historical
    data already in memory (the `daily` DataFrame) -- that table is NOT
    saved in the pickle, so for a backend/live use case you must supply
    the count yourself (e.g. from a live violations feed, or a manual
    count for "today so far").
    """
    baselines = bundle["anomaly_baselines"]
    threshold = bundle["anomaly_threshold"]

    if cluster_id not in baselines:
        return {"cluster_id": cluster_id, "error": "cluster not found"}

    stats = baselines[cluster_id]
    mean = stats["baseline_mean"]
    std = stats["baseline_std"]

    if std and std > 0:
        z = (actual_count - mean) / std
    else:
        z = 0.0  # no meaningful baseline -- matches notebook's own handling

    return {
        "cluster_id": int(cluster_id),
        "date": str(date),
        "actual_count": int(actual_count),
        "baseline_mean": round(float(mean), 2),
        "z_score": round(float(z), 2),
        "is_anomalous": bool(z > threshold),
    }
```

**Difference from the notebook's version, and why:** the notebook's
`check_anomaly` looks up `actual_count` automatically from the `daily`
DataFrame (built from historical data already loaded in the notebook).
That DataFrame is **not** part of the pickle — only the summary
statistics (`baseline_mean`, `baseline_std`) are. So for the backend, you
must supply `actual_count` yourself as an argument. For a *historical*
date already in your dataset, your backend would need to look that count
up from wherever your violations are stored (e.g. a database query). For
"today," you'd pass in today's running tally as it comes in.

### Output (exact shape)

```json
{
  "cluster_id": 0,
  "date": "2023-12-22",
  "actual_count": 47,
  "baseline_mean": 8.05,
  "z_score": 3.31,
  "is_anomalous": true
}
```

`is_anomalous` is `true` when `z_score > 2.5` (`bundle["anomaly_threshold"]`).

### Why this one is the genuinely "live" function

Unlike Tier 2, this isn't limited to a small precomputed set of answers —
`actual_count` can be any number, including one that's never happened
before (e.g. a live count for today, mid-day, that doesn't exist in any
historical table yet). This is the function to use for a demo moment
where you want to show something the static JSON genuinely couldn't have
precomputed: feed it a live/incoming count and watch it compute a fresh
z-score on the spot.

---

## Quick reference: full request → response example

**Request:** "Is hotspot 2 normally risky on Monday at 10am, and was
anything unusual there on 2023-12-22?"

```python
predict_priority_risk(cluster_id=2, day_of_week="Monday", hour=10,
                       bundle=bundle, cluster_summary=cluster_summary)
# -> {"cluster_id": 2, "day_of_week": "Monday", "hour": 10,
#     "time_bucket": "late_morning", "predicted_risk": "HIGH",
#     "confidence": 0.861}

check_anomaly(cluster_id=0, date="2023-12-22", actual_count=47, bundle=bundle)
# -> {"cluster_id": 0, "date": "2023-12-22", "actual_count": 47,
#     "baseline_mean": 8.05, "z_score": 3.31, "is_anomalous": true}
```

---

## Common mistakes to avoid

- **Don't pass lowercase or abbreviated day names** (`"mon"`, `"monday"`) —
  must exactly match `bundle["day_map"]` keys, which are capitalized full
  names.
- **Don't pass a time bucket string directly as `hour`** — `hour` is an
  integer 0–23; bucketing happens automatically inside the function.
- **Don't reload the pickle on every request** — load once at startup,
  reuse the in-memory `bundle` object.
- **Don't expect `check_anomaly` to work without `actual_count`** — unlike
  the notebook version, the standalone backend version needs you to supply
  the observed count; the pickle does not contain historical daily counts.
- **Don't expect new answers from `predict_priority_risk`** — it's
  deterministic over a fixed, small input space (300 × 7 × 5); every
  possible output is already sitting in `hotspots_details.json`.
